# Test: Generate 5 Lensed + 5 Non-Lensed Images
Self-contained test using real JWST F115W backgrounds. No CosmoDC2 or VELA required.
Runs on Apple Silicon via numpy/Accelerate + multiprocessing across M3 cores.

In [1]:
import os
# Disable numba JIT — needed on macOS with Homebrew Python due to scipy/numba conflict
os.environ['NUMBA_DISABLE_JIT'] = '1'

import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for nbconvert
import matplotlib.pyplot as plt

from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.Cosmo.lens_cosmo import LensCosmo

# ── Paths ──────────────────────────────────────────────────────────────────
PREPPED_DIR = "prepped"

# ── Image config ───────────────────────────────────────────────────────────
PIXELS     = 125
PIXEL_SIZE = 0.031        # arcsec/pix — NIRCam SW channel
EXP_TIME   = 1380

# ── Real JWST backgrounds ──────────────────────────────────────────────────
sum_to_flux  = 6.501853565914121
PIXAR_SR     = 2.29232933396454e-14
mjysr_to_sim = PIXAR_SR * 1e15 * sum_to_flux   # ≈ 149.0

real_bgs     = np.load(os.path.join(PREPPED_DIR, "real_backgrounds.npy"))
real_bgs_sim = (real_bgs * mjysr_to_sim).astype(np.float32)
print(f"Loaded {len(real_bgs_sim)} real JWST background patches.")
print(f"Unit scale factor: {mjysr_to_sim:.2f}  (MJy/sr → sim units)")

rng_bg = np.random.default_rng()

def get_real_background():
    return real_bgs_sim[rng_bg.integers(len(real_bgs_sim))]

# ── Sampling helpers (no scipy needed) ────────────────────────────────────

def truncnorm_rvs(a, b, loc, scale, rng, max_tries=1000):
    for _ in range(max_tries):
        x = rng.normal(loc, scale)
        if a <= (x - loc) / scale <= b:
            return float(x)
    return float(loc + 0.5 * (a + b) * scale)

def uniform_rvs(low, high, rng):
    return float(rng.uniform(low, high))

print("Ready.")

Loaded 5000 real JWST background patches.
Unit scale factor: 149.04  (MJy/sr → sim units)
Ready.


In [2]:
# ── Mass / stellar-mass helpers (unchanged from original notebook) ──────────

def stellar_mass(M, z):
    mM10=11.88; mu=0.019; mM00=0.0282; nu=-0.72
    gamma0=0.556; gamma1=-0.26; beta0=1.06; beta1=0.17
    M1 = 10**(11.88*(z+1)**mu)
    mM0 = mM00*(z+1)**nu
    gamma = gamma0*(z+1)**gamma1
    beta  = beta1*z + beta0
    shmr  = 2*mM0/((M/M1)**(-beta)+(M/M1)**gamma)
    return shmr * M

def ML_ratio(z):
    return 10**(2.15259223299506*np.log10(z)+6.61731435158865)

# ── Lenstronomy setup helpers ──────────────────────────────────────────────

def make_kwargs_data():
    return {
        'background_rms': 0,
        'exposure_time': EXP_TIME,
        'ra_at_xy_0':  -PIXELS/2 * PIXEL_SIZE,
        'dec_at_xy_0': -PIXELS/2 * PIXEL_SIZE,
        'transform_pix2angle': np.array([[PIXEL_SIZE, 0.], [0., PIXEL_SIZE]]),
        'image_data': np.zeros((PIXELS, PIXELS))
    }

def make_psf():
    # Gaussian PSF — matches original notebook
    return PSF(psf_type='GAUSSIAN', fwhm=0.06, pixel_size=PIXEL_SIZE, truncation=3)

print("Helpers defined.")

Helpers defined.


In [3]:
# ── simulate_one ──────────────────────────────────────────────────────────

def simulate_one(lensed=True, seed=None):
    rng = np.random.default_rng(seed)

    # --- Redshifts ---
    z_source = truncnorm_rvs(-1/3, 2/3, loc=1.5, scale=3, rng=rng)
    z_source = max(z_source, 0.3)
    z_lens   = uniform_rvs(0.1, min(0.8, z_source - 0.1), rng)
    z_lens   = max(0.05, min(z_lens, z_source * 0.8))

    # --- Halo mass → velocity dispersion → Einstein radius ---
    log_mass = uniform_rvs(13.0, 14.5, rng)
    mass     = 10**log_mass
    mStar    = stellar_mass(mass, z_lens)

    alpha = 0.16; beta_sig = 3.31
    scatter = float(rng.normal(0, 0.17))
    z_sig   = 100 * 10**((np.log10(mass / 1e12) - alpha + scatter) / beta_sig)

    lens_cosmo = LensCosmo(z_lens, z_source)
    theta_E    = lens_cosmo.sis_sigma_v2theta_E(z_sig) if lensed else 0.0

    # Enforce .5–1.5″ range for lensed
    if lensed and not (0.5 <= theta_E <= 1.5):
        return simulate_one(lensed=lensed, seed=int(rng.integers(int(1e9))))

    # --- Galaxy parameters ---
    e1, e2        = rng.normal(0, 0.15, size=2).clip(-0.5, 0.5)
    R_sersic_lens = truncnorm_rvs(0, 3, loc=0.3,  scale=0.3,  rng=rng)
    n_sersic_lens = uniform_rvs(2, 6, rng)
    R_sersic_src  = truncnorm_rvs(0, 3, loc=0.15, scale=0.15, rng=rng)
    n_sersic_src  = uniform_rvs(1, 4, rng)
    e1s, e2s      = rng.normal(0, 0.2, size=2).clip(-0.6, 0.6)
    center_x, center_y = rng.normal(0, 0.25, size=2)

    # --- Lenstronomy setup ---
    kwargs_data     = make_kwargs_data()
    kwargs_numerics = {'supersampling_factor': 1, 'supersampling_convolution': False}
    data_class      = ImageData(**kwargs_data)
    psf_class       = make_psf()

    source_model_class     = LightModel(['SERSIC_ELLIPSE'])
    lens_light_model_class = LightModel(['SERSIC_ELLIPSE'])
    lens_model_class       = LensModel(['SIE'], z_lens=z_lens)

    image_model = ImageModel(
        data_class=data_class, psf_class=psf_class,
        lens_model_class=lens_model_class,
        source_model_class=source_model_class,
        lens_light_model_class=lens_light_model_class,
        kwargs_numerics=kwargs_numerics
    )

    kwargs_lens       = [{'theta_E': theta_E, 'e1': e1, 'e2': e2,
                          'center_x': 0.0, 'center_y': 0.0}]
    kwargs_lens_light = [{'amp': 1, 'R_sersic': R_sersic_lens,
                          'n_sersic': n_sersic_lens, 'e1': e1, 'e2': e2,
                          'center_x': 0.0, 'center_y': 0.0}]
    kwargs_source     = [{'amp': 1, 'R_sersic': R_sersic_src,
                          'n_sersic': n_sersic_src, 'e1': e1s, 'e2': e2s,
                          'center_x': center_x, 'center_y': center_y}]

    # --- Calibrate source amplitude (matching original notebook scale_up) ---
    scale_up     = 10**uniform_rvs(0, 2, rng)          # random arc brightness boost
    src_flux_njy = truncnorm_rvs(0, 3, loc=50, scale=80, rng=rng)
    calc_sum_src = sum_to_flux * src_flux_njy

    img_src = image_model.image(kwargs_lens, kwargs_source,
                                kwargs_lens_light=kwargs_lens_light, kwargs_ps=None,
                                source_add=True, lens_light_add=False)
    s = np.sum(img_src)
    if s <= 0:
        return simulate_one(lensed=lensed, seed=int(rng.integers(int(1e9))))
    kwargs_source[0]['amp'] = (calc_sum_src / s) * scale_up

    # Non-lensed: no source visible (matches original notebook for .5-1.5" size)
    if not lensed:
        kwargs_source[0]['amp'] = 0.0

    # --- Calibrate lens light amplitude ---
    lStar         = mStar / ML_ratio(max(z_lens, 0.01))
    calc_sum_lens = sum_to_flux * lStar
    img_lens = image_model.image(kwargs_lens, kwargs_source,
                                 kwargs_lens_light=kwargs_lens_light, kwargs_ps=None,
                                 source_add=False, lens_light_add=True)
    s = np.sum(img_lens)
    if s <= 0:
        return simulate_one(lensed=lensed, seed=int(rng.integers(int(1e9))))
    kwargs_lens_light[0]['amp'] = calc_sum_lens / s

    # --- Final images ---
    image = image_model.image(kwargs_lens, kwargs_source,
                              kwargs_lens_light=kwargs_lens_light, kwargs_ps=None,
                              source_add=True, lens_light_add=True)

    # Arc-only image (source with lensing, no lens galaxy light) — for diagnostics
    image_source = image_model.image(kwargs_lens, kwargs_source,
                                     kwargs_lens_light=kwargs_lens_light, kwargs_ps=None,
                                     source_add=True, lens_light_add=False)

    return image, image_source, theta_E, z_lens, z_source, mass, mStar

print("simulate_one() defined.")

simulate_one() defined.


In [4]:
# ── Generate 5 non-lensed + 5 lensed sequentially ─────────────────────────
# numpy uses Apple Accelerate (vecLib) automatically on M-series — no extra
# setup needed. Multiprocessing is skipped since it doesn't work in notebooks
# on macOS; for large runs use the standalone prep_jwst.py script instead.

import time

N_EACH   = 5
N_TOTAL  = N_EACH * 2

images        = np.zeros((N_TOTAL, PIXELS, PIXELS), dtype=np.float32)
image_sources = np.zeros_like(images)
labels        = np.zeros(N_TOTAL)
theta_Es      = np.zeros(N_TOTAL)
z_lenses      = np.zeros(N_TOTAL)
z_sources     = np.zeros(N_TOTAL)
masses        = np.zeros(N_TOTAL)

rng_main = np.random.default_rng(42)
jobs = [(i, False) for i in range(N_EACH)] + [(i + N_EACH, True) for i in range(N_EACH)]

t0 = time.time()
for idx, lensed in jobs:
    label = "lensed" if lensed else "non-lensed"
    print(f"  [{idx+1}/{N_TOTAL}] Generating {label}...", end=" ", flush=True)
    t1 = time.time()

    result = simulate_one(lensed=lensed, seed=int(rng_main.integers(int(1e9))))
    image, image_source, theta_E, z_lens, z_source, mass, mStar = result

    bg = get_real_background()
    images[idx]        = image + bg
    image_sources[idx] = image_source + bg
    labels[idx]        = 1.0 if lensed else 0.0
    theta_Es[idx]      = theta_E
    z_lenses[idx]      = z_lens
    z_sources[idx]     = z_source
    masses[idx]        = mass

    print(f"{time.time()-t1:.1f}s")

print(f"\nDone — total time: {time.time()-t0:.1f}s")
print(f"  Non-lensed: {int((labels==0).sum())}   Lensed: {int((labels==1).sum())}")

  [1/10] Generating non-lensed... 

0.0s
  [2/10] Generating non-lensed... 

0.0s
  [3/10] Generating non-lensed... 

0.0s
  [4/10] Generating non-lensed... 

0.0s
  [5/10] Generating non-lensed... 

...

0.0s
  [7/10] Generating lensed... 

0.0s
  [8/10] Generating lensed... 

0.0s
  [9/10] Generating lensed... 

0.0s
  [10/10] Generating lensed... 

0.0s

Done — total time: 0.0s
  Non-lensed: 5   Lensed: 5


In [5]:
# ── Save outputs ───────────────────────────────────────────────────────────
OUT = "output"
os.makedirs(OUT, exist_ok=True)

np.save(f"{OUT}/images.npy",       images)
np.save(f"{OUT}/lensed.npy",       labels)
np.save(f"{OUT}/theta_Es.npy",     theta_Es)
np.save(f"{OUT}/z_lens.npy",       z_lenses)
np.save(f"{OUT}/z_source.npy",     z_sources)
np.save(f"{OUT}/masses.npy",       masses)
np.save(f"{OUT}/image_source.npy", image_sources)

print(f"Saved to {OUT}/")
for f in os.listdir(OUT):
    path = os.path.join(OUT, f)
    print(f"  {f:<25} {os.path.getsize(path)/1e3:.1f} KB")

Saved to output/
  z_source.npy              0.2 KB
  preview.png               241.2 KB
  theta_Es.npy              0.2 KB
  images.npy                625.1 KB
  lensed.npy                0.2 KB
  masses.npy                0.2 KB
  z_lens.npy                0.2 KB
  image_source.npy          625.1 KB


In [6]:
# ── Visualise ─────────────────────────────────────────────────────────────
# Row 1: Non-lensed full images
# Row 2: Lensed full images
# Row 3: Lensed arc-only (no lens light) — confirms arcs are present

order_nl = np.where(labels == 0)[0]   # non-lensed indices
order_l  = np.where(labels == 1)[0]   # lensed indices

fig, axs = plt.subplots(3, N_EACH, figsize=(20, 13), dpi=100)

def show(ax, img, title, cmap='gray', sqrt=False):
    d = np.sqrt(np.clip(img, 0, None)) if sqrt else img
    vmin, vmax = np.percentile(d, 1), np.percentile(d, 99.5)
    im = ax.imshow(d, vmin=vmin, vmax=vmax, origin='lower', cmap=cmap)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=8)
    ax.axis('off')

# Row 0: non-lensed (sqrt stretch to show faint structure)
for col, idx in enumerate(order_nl):
    zl = z_lenses[idx]
    show(axs[0, col], images[idx],
         f"non-lensed  zl={zl:.2f}", sqrt=True)

# Row 1: lensed full image (sqrt stretch)
for col, idx in enumerate(order_l):
    tE = theta_Es[idx]; zl = z_lenses[idx]; zs = z_sources[idx]
    show(axs[1, col], images[idx],
         f"lensed θE={tE:.2f}\"  zl={zl:.2f}  zs={zs:.2f}", sqrt=True)

# Row 2: lensed arc-only (no lens light) — linear stretch
for col, idx in enumerate(order_l):
    tE = theta_Es[idx]
    arc = image_sources[idx]
    show(axs[2, col], arc,
         f"arcs only  θE={tE:.2f}\"", cmap='inferno', sqrt=False)

axs[0, 0].set_ylabel("Non-lensed\n(√ stretch)", fontsize=11)
axs[1, 0].set_ylabel("Lensed\n(√ stretch)", fontsize=11)
axs[2, 0].set_ylabel("Arc-only\n(linear)", fontsize=11)

plt.suptitle('Real JWST F115W backgrounds + lenstronomy sims  (jw01810 o002)', fontsize=13)
plt.tight_layout()
plt.savefig('output/preview.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → output/preview.png")

Saved → output/preview.png


/var/folders/5c/vztf2q9s5272j5sc3t9b212w0000gn/T/ipykernel_61520/856001939.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
